# Step 3: YAMNet from Kaggle, Embedding Extraction, and Prototype Inference

This notebook uses the pretrained YAMNet model from Kaggle as the frozen embedding backbone. The notebook downloads the model with Kaggle tooling, extracts 1024-dimensional embeddings, builds a prototype store from training clips, evaluates nearest-prototype inference, and exports the resulting artifacts.

Before running the download cell, make sure Kaggle credentials are configured in the notebook environment.

In [1]:
%pip install -q -e ..

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import kagglehub
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
from sklearn.metrics import classification_report

PROJECT_ROOT = Path.cwd().resolve().parents[1]
ML_ROOT = PROJECT_ROOT / 'ml'
SRC_ROOT = ML_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from soundpulse_ml.datasets import load_dataset_manifest, manifest_to_dataframe

CONTRACTS_DIR = PROJECT_ROOT / 'packages' / 'shared_contracts' / 'v1'
CURATED_MANIFEST_PATH = ML_ROOT / 'data' / 'curated' / 'soundpulse_manifest.v1.json'
YAMNET_OUTPUT_DIR = ML_ROOT / 'export' / 'artifacts' / 'generated' / 'yamnet_kaggle'
YAMNET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with (CONTRACTS_DIR / 'class_labels.json').open('r', encoding='utf-8') as handle:
    CLASS_CONTRACT = json.load(handle)
with (CONTRACTS_DIR / 'preprocessing_config.json').open('r', encoding='utf-8') as handle:
    PREPROCESSING_CONTRACT = json.load(handle)

CLASS_ORDER = [item['slug'] for item in CLASS_CONTRACT['classes']]
CLASS_TO_INDEX = {slug: index for index, slug in enumerate(CLASS_ORDER)}
SAMPLE_RATE = PREPROCESSING_CONTRACT['audio']['sample_rate_hz']
WINDOW_DURATION_MS = PREPROCESSING_CONTRACT['audio']['window_duration_ms']
WINDOW_SAMPLES = int(SAMPLE_RATE * WINDOW_DURATION_MS / 1000)
YAMNET_KAGGLE_HANDLE = os.environ.get('SOUNDPULSE_YAMNET_KAGGLE_HANDLE', 'google/yamnet/tensorFlow2/yamnet')
SHARED_TEST_CAP = 64
MAX_EXAMPLES_PER_CLASS_BY_SPLIT = {'train': 128, 'validation': 64, 'test': SHARED_TEST_CAP}

assert CURATED_MANIFEST_PATH.exists(), 'Run Notebook 1 first to build the curated manifest.'
print({
    'yamnet_kaggle_handle': YAMNET_KAGGLE_HANDLE,
    'sample_rate_hz': SAMPLE_RATE,
    'max_examples_per_class_by_split': MAX_EXAMPLES_PER_CLASS_BY_SPLIT,
})

{'yamnet_kaggle_handle': 'google/yamnet/tensorFlow2/yamnet', 'sample_rate_hz': 16000, 'max_examples_per_class_by_split': {'train': 128, 'validation': 64, 'test': 64}}


In [3]:
import json

# Path to the config file
config_path = os.path.expanduser('~/.kaggle/kaggle.json')

if os.path.exists(config_path):
    with open(config_path) as f:
        data = json.load(f)
        os.environ['KAGGLE_USERNAME'] = data['username']
        os.environ['KAGGLE_KEY'] = data['key']
    print("Environment variables successfully set from kaggle.json")
else:
    print("kaggle.json not found in ~/.kaggle/")

Environment variables successfully set from kaggle.json


In [5]:
if not os.environ.get('KAGGLE_USERNAME') or not os.environ.get('KAGGLE_KEY'):
    raise EnvironmentError(
        'Kaggle credentials are not set in this environment. Set KAGGLE_USERNAME and KAGGLE_KEY before running the YAMNet download cell.'
    )

yamnet_download_dir = Path(kagglehub.model_download(YAMNET_KAGGLE_HANDLE))
yamnet_download_dir

100%|██████████| 13.8k/13.8k [00:00<00:00, 5.05MB/s]



100%|██████████| 7.23k/7.23k [00:00<?, ?B/s]





100%|██████████| 3.03M/3.03M [00:08<00:00, 382kB/s]













100%|██████████| 14.4M/14.4M [00:31<00:00, 479kB/s]


WindowsPath('C:/Users/ZBOOK STUDIO G5/.cache/kagglehub/models/google/yamnet/tensorFlow2/yamnet/1')

In [6]:
def find_yamnet_artifact(model_root: Path) -> Path:
    candidates = list(model_root.rglob('saved_model.pb')) + list(model_root.rglob('*.keras')) + list(model_root.rglob('*.h5'))
    if not candidates:
        raise FileNotFoundError(f'No loadable YAMNet artifact found under {model_root}')
    return candidates[0]


yamnet_artifact = find_yamnet_artifact(yamnet_download_dir)
yamnet_artifact

WindowsPath('C:/Users/ZBOOK STUDIO G5/.cache/kagglehub/models/google/yamnet/tensorFlow2/yamnet/1/saved_model.pb')

In [7]:
def load_yamnet_model(artifact_path: Path):
    if artifact_path.name == 'saved_model.pb':
        model_root = artifact_path.parent
        saved_model = tf.saved_model.load(str(model_root))
        return saved_model
    return tf.keras.models.load_model(str(artifact_path))


yamnet_model = load_yamnet_model(yamnet_artifact)
type(yamnet_model)

tensorflow.python.saved_model.load.Loader._recreate_base_user_object.<locals>._UserObject

In [8]:
manifest = load_dataset_manifest(CURATED_MANIFEST_PATH)
manifest_df = manifest_to_dataframe(manifest)
manifest_df = manifest_df[manifest_df['audio_path'].map(lambda value: Path(value).exists())].reset_index(drop=True)

def cap_examples_per_class(frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
    split_frame = frame[frame['split'] == split_name].copy()
    max_examples = MAX_EXAMPLES_PER_CLASS_BY_SPLIT.get(split_name)
    if max_examples is None:
        return split_frame.reset_index(drop=True)
    return (
        split_frame
        .sort_values(['class_slug', 'clip_id'])
        .groupby('class_slug', group_keys=False)
        .head(max_examples)
        .reset_index(drop=True)
    )

manifest_df = pd.concat(
    [cap_examples_per_class(manifest_df, split_name) for split_name in ['train', 'validation', 'test']],
    ignore_index=True,
)

manifest_df.groupby(['split', 'class_slug']).size().unstack(fill_value=0)

class_slug,background,clock_alarm,crying_baby,dog_barking,door_knock,door_wood_creaking,doorbell,fire_alarm,speech,water_running
split,,,,,,,,,,
test,64,8,38,64,64,64,34,64,64,64
train,128,24,114,128,128,128,90,128,128,128
validation,64,8,25,64,50,62,17,64,64,40


In [9]:
def load_audio_file(audio_path: str) -> np.ndarray:
    waveform, sample_rate = sf.read(audio_path, dtype='float32', always_2d=False)
    if waveform.ndim > 1:
        waveform = np.mean(waveform, axis=1)
    if sample_rate != SAMPLE_RATE:
        waveform = librosa.resample(waveform, orig_sr=sample_rate, target_sr=SAMPLE_RATE)
    return waveform.astype(np.float32)


def resolve_embedding_tensor(output: object) -> np.ndarray:
    if isinstance(output, dict):
        for value in output.values():
            value_np = np.array(value)
            if value_np.ndim >= 2 and value_np.shape[-1] == 1024:
                return value_np
    if isinstance(output, (list, tuple)):
        for value in output:
            value_np = np.array(value)
            if value_np.ndim >= 2 and value_np.shape[-1] == 1024:
                return value_np
    value_np = np.array(output)
    if value_np.ndim >= 2 and value_np.shape[-1] == 1024:
        return value_np
    raise ValueError('Unable to resolve a 1024-d embedding tensor from the YAMNet output')


def extract_yamnet_embedding(waveform: np.ndarray) -> np.ndarray:
    input_tensor = tf.convert_to_tensor(waveform, dtype=tf.float32)
    if hasattr(yamnet_model, 'signatures') and yamnet_model.signatures:
        signature = next(iter(yamnet_model.signatures.values()))
        output = signature(input_tensor)
    else:
        output = yamnet_model(input_tensor)
    embedding_frames = resolve_embedding_tensor(output)
    if embedding_frames.ndim == 1:
        embedding_frames = np.expand_dims(embedding_frames, axis=0)
    return embedding_frames.mean(axis=0).astype(np.float32)

In [10]:
embedding_rows = []
for row in manifest_df.to_dict(orient='records'):
    waveform = load_audio_file(row['audio_path'])
    embedding = extract_yamnet_embedding(waveform)
    embedding_rows.append({
        'clip_id': row['clip_id'],
        'class_slug': row['class_slug'],
        'split': row['split'],
        'embedding': embedding,
    })

embedding_df = pd.DataFrame(embedding_rows)
embedding_df.head()

,clip_id,class_slug,split,embedding
0,fsd50k-100005,background,train,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,fsd50k-100006,background,train,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,fsd50k-100007,background,train,"[0.0, 0.0, 0.008183795, 0.0, 0.0, 0.0, 0.0, 0...."
3,fsd50k-100762,background,train,"[0.0036029757, 0.09971703, 0.07399158, 0.16090..."
4,fsd50k-101135,background,train,"[0.3295711, 0.09041548, 0.10840413, 0.44236222..."


In [11]:
train_embeddings = embedding_df[embedding_df['split'] == 'train']
prototype_store = {}
for class_slug, class_frame in train_embeddings.groupby('class_slug'):
    prototype_store[class_slug] = np.stack(class_frame['embedding'].to_list(), axis=0).mean(axis=0).tolist()

with (YAMNET_OUTPUT_DIR / 'prototype_store.json').open('w', encoding='utf-8') as handle:
    json.dump(prototype_store, handle, indent=2)

print(f'Exported prototypes for {len(prototype_store)} classes')

Exported prototypes for 10 classes


In [12]:
def prototype_predict(embedding: np.ndarray, prototypes: dict[str, list[float]]) -> tuple[str, float]:
    distances = {
        class_slug: float(np.sum((embedding - np.array(prototype, dtype=np.float32)) ** 2))
        for class_slug, prototype in prototypes.items()
    }
    predicted_label = min(distances, key=distances.get)
    return predicted_label, distances[predicted_label]

test_rows = embedding_df[embedding_df['split'] == 'test']
predictions = []
for row in test_rows.to_dict(orient='records'):
    predicted_label, distance = prototype_predict(row['embedding'], prototype_store)
    predictions.append({
        'clip_id': row['clip_id'],
        'true_label': row['class_slug'],
        'predicted_label': predicted_label,
        'distance': distance,
    })

predictions_df = pd.DataFrame(predictions)
print(
    classification_report(
        predictions_df['true_label'],
        predictions_df['predicted_label'],
        labels=CLASS_ORDER,
        zero_division=0,
    )
)
predictions_df.head()

                    precision    recall  f1-score   support

        fire_alarm       0.49      0.30      0.37        64
       clock_alarm       0.39      0.88      0.54         8
        door_knock       0.25      0.25      0.25        64
          doorbell       0.30      0.38      0.33        34
       crying_baby       0.97      0.74      0.84        38
     water_running       0.93      0.80      0.86        64
       dog_barking       1.00      0.56      0.72        64
door_wood_creaking       0.37      0.47      0.41        64
            speech       0.38      0.83      0.52        64
        background       0.60      0.23      0.34        64

          accuracy                           0.51       528
         macro avg       0.57      0.54      0.52       528
      weighted avg       0.58      0.51      0.51       528



,clip_id,true_label,predicted_label,distance
0,fsd50k-100355,background,speech,34.465714
1,fsd50k-101762,background,door_knock,70.728767
2,fsd50k-101765,background,fire_alarm,75.687599
3,fsd50k-102542,background,background,30.037247
4,fsd50k-102780,background,background,35.425034


In [13]:
artifact_summary = {
    'yamnet_kaggle_handle': YAMNET_KAGGLE_HANDLE,
    'yamnet_artifact': str(yamnet_artifact),
    'embedding_dim': PREPROCESSING_CONTRACT['embedding_contract']['yamnet_dim'],
    'class_order': CLASS_ORDER,
}

with (YAMNET_OUTPUT_DIR / 'artifact_summary.json').open('w', encoding='utf-8') as handle:
    json.dump(artifact_summary, handle, indent=2)

predictions_df.to_csv(YAMNET_OUTPUT_DIR / 'prototype_predictions.csv', index=False)
print(f'Wrote YAMNet outputs to {YAMNET_OUTPUT_DIR}')

Wrote YAMNet outputs to C:\Users\ZBOOK STUDIO G5\Documents\GitHub\SoundPulse\ml\export\artifacts\generated\yamnet_kaggle


In [14]:
sample_row = manifest_df.iloc[0]
sample_waveform = load_audio_file(sample_row['audio_path'])
sample_embedding = extract_yamnet_embedding(sample_waveform)
sample_label, sample_distance = prototype_predict(sample_embedding, prototype_store)
print({'clip_id': sample_row['clip_id'], 'true_label': sample_row['class_slug'], 'predicted_label': sample_label, 'distance': sample_distance})

{'clip_id': 'fsd50k-100005', 'true_label': 'background', 'predicted_label': 'speech', 'distance': 34.31035614013672}
